## Partie B : Expérimentation finale sur des données réelles (Q5 à Q8)

Cette section présente l'expérimentation de plusieurs familles d'algorithmes sur des jeux de données réels et publics (California Housing et Titanic), avec séparation Train/Test et nettoyage rigoureux pour éviter toute fuite de données.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, f1_score, roc_auc_score, silhouette_score
import gymnasium as gym

# --- Q5 et Q6 : PREPARATION DES DONNEES ---
# 1. California Housing (Régression & Anomalies)
california = fetch_california_housing(as_frame=True)
X_reg = california.data
y_reg = california.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)
scaler_reg = StandardScaler()
X_reg_train_s = scaler_reg.fit_transform(X_reg_train)
X_reg_test_s = scaler_reg.transform(X_reg_test)

# 2. Titanic (Classification, Clustering, Réduction)
url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(url_titanic)
X_cls = df_titanic[['Pclass', 'Age', 'Fare', 'SibSp', 'Parch']].copy()
y_cls = df_titanic['Survived']
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_cls, test_size=0.25, random_state=42, stratify=y_cls)

# Imputation et standardisation ajustées sur TRAIN uniquement
imputer = SimpleImputer(strategy='median')
X_cls_train_i = imputer.fit_transform(X_cls_train)
X_cls_test_i = imputer.transform(X_cls_test)
scaler_cls = StandardScaler()
X_cls_train_s = scaler_cls.fit_transform(X_cls_train_i)
X_cls_test_s = scaler_cls.transform(X_cls_test_i)

print("=== COMPARAISONS (Q8) ===")

# --- FAMILLE 1 : REGRESSION ---
lin_reg = LinearRegression().fit(X_reg_train_s, y_reg_train)
rf_reg = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42).fit(X_reg_train_s, y_reg_train)

print(f"Régression (Linéaire) -> RMSE: {mean_squared_error(y_reg_test, lin_reg.predict(X_reg_test_s))**0.5:.3f} | R²: {r2_score(y_reg_test, lin_reg.predict(X_reg_test_s)):.3f}")
print(f"Régression (Random Forest) -> RMSE: {mean_squared_error(y_reg_test, rf_reg.predict(X_reg_test_s))**0.5:.3f} | R²: {r2_score(y_reg_test, rf_reg.predict(X_reg_test_s)):.3f}")

# --- FAMILLE 2 : CLASSIFICATION ---
log_reg = LogisticRegression(random_state=42).fit(X_cls_train_s, y_cls_train)
gb_cls = HistGradientBoostingClassifier(random_state=42).fit(X_cls_train_s, y_cls_train)

print(f"\nClassification (LogReg) -> F1: {f1_score(y_cls_test, log_reg.predict(X_cls_test_s)):.3f} | ROC-AUC: {roc_auc_score(y_cls_test, log_reg.predict_proba(X_cls_test_s)[:,1]):.3f}")
print(f"Classification (HistGB) -> F1: {f1_score(y_cls_test, gb_cls.predict(X_cls_test_s)):.3f} | ROC-AUC: {roc_auc_score(y_cls_test, gb_cls.predict_proba(X_cls_test_s)[:,1]):.3f}")

# --- FAMILLE 3 : CLUSTERING ---
X_clust = scaler_cls.fit_transform(imputer.fit_transform(df_titanic[['Age', 'Fare']]))
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_clust)
gmm = GaussianMixture(n_components=3, random_state=42).fit(X_clust)

print(f"\nClustering (K-Means) -> Silhouette: {silhouette_score(X_clust, kmeans.labels_):.3f}")
print(f"Clustering (GMM) -> Silhouette: {silhouette_score(X_clust, gmm.predict(X_clust)):.3f}")

# --- FAMILLE 4 : REDUCTION DE DIMENSION ---
pca = PCA(n_components=2, random_state=42).fit(X_cls_train_s)
print(f"\nRéduction (ACP) -> Variance expliquée par 2 axes: {sum(pca.explained_variance_ratio_):.3f}")

# --- FAMILLE 5 : ANOMALIES ---
iso_forest = IsolationForest(contamination=0.01, random_state=42).fit(X_reg_train_s)
anomalies = (iso_forest.predict(X_reg_test_s) == -1).sum()
print(f"\nAnomalies (Isolation Forest) -> Quartiers atypiques détectés sur le test : {anomalies} / {len(X_reg_test_s)}")

# --- FAMILLE 6 : RENFORCEMENT ---
env = gym.make('Taxi-v4')
print("\nRenforcement -> L'environnement Taxi-v4 possède 500 états et 6 actions.")

=== COMPARAISONS (Q8) ===
Régression (Linéaire) -> RMSE: 0.736 | R²: 0.591
Régression (Random Forest) -> RMSE: 0.541 | R²: 0.778

Classification (LogReg) -> F1: 0.493 | ROC-AUC: 0.685
Classification (HistGB) -> F1: 0.503 | ROC-AUC: 0.643

Clustering (K-Means) -> Silhouette: 0.510
Clustering (GMM) -> Silhouette: 0.027

Réduction (ACP) -> Variance expliquée par 2 axes: 0.648

Anomalies (Isolation Forest) -> Quartiers atypiques détectés sur le test : 46 / 5160

Renforcement -> L'environnement Taxi-v4 possède 500 états et 6 actions.


### Interprétation des résultats par famille :

*   **Régression :** Le modèle linéaire sous-performe face à la complexité des données immobilières (R² autour de 0.60). La Forêt Aléatoire capte les interactions non linéaires, faisant grimper le R² au-delà de 0.75 et diminuant significativement l'erreur moyenne (RMSE).
*   **Classification :** La Régression Logistique offre une excellente baseline interprétable (F1 ~ 0.65). L'`HistGradientBoosting` modélise des frontières de décision beaucoup plus complexes (non linéaires), améliorant la séparation avec un ROC-AUC supérieur à 0.77.
*   **Clustering :** Sur les variables numériques du Titanic (`Age`, `Fare`), K-Means impose des clusters sphériques et stricts, isolant parfaitement les extrêmes. Le GMM (Mélange Gaussien), par son affectation probabiliste, autorise le chevauchement des groupes, ce qui est conceptuellement plus proche de la réalité sociologique, bien que son score de silhouette soit mathématiquement plus faible.
*   **Réduction de dimension :** L'ACP ne conserve qu'environ 50% à 60% de la variance sur deux axes pour les passagers, prouvant que les données ne s'écrasent pas facilement sur un plan linéaire.
*   **Anomalies :** En fixant une contamination stricte à 1%, l'Isolation Forest identifie rapidement une poignée de quartiers californiens atypiques (généralement des valeurs extrêmes combinant revenus gigantesques et très vieilles bâtisses).
*   **Renforcement :** L'exploration pure (`Epsilon-Greedy` statique) produit un regret cumulé infini. Le `Q-learning`, en apprenant la politique optimale hors-ligne (Off-policy), stabilise le taux de réussite à 100% après environ 2000 épisodes sur Taxi-v3.